# 01 — Straddle payoff vs profit, with real SPY prices

**Question:** if you buy or sell the Oct 23, 771-strike SPY straddle at Friday's closing price, what do you make or lose at expiry for each possible SPY price?

What is real and what is assumed:
- **Real (observed):** call and put bid/ask quotes from the Yahoo Finance snapshot taken Fri 2026-09-25 at 16:14 ET, 14 minutes after the close.
- **Assumption:** we treat the **midpoint** of bid and ask as the price paid. A real trade would usually get a slightly worse price.
- **Ignored for now:** commissions, early exercise (SPY options are American-style) and the time value of money.
- **Hypothetical:** SPY prices at expiry. The x-axis covers possible outcomes, not a forecast.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from options_lab.payoffs import position_profit, straddle_payoff

SNAPSHOT = "../data/raw/spy_chain_20260925T201452Z.csv"
EXPIRY = "2026-10-23"
STRIKE = 771.0
SPY_AT_SNAPSHOT = 771.30  # from the snapshot's .json file (15:59 ET 1-minute bar)

## Step 1 — Find the two contracts in the snapshot

`chain` is a pandas **DataFrame**: a table where each row is one option contract and each column is a field such as `strike` or `bid`.

`chain[condition]` keeps only the rows where the condition is `True`. You combine conditions with `&` ("and"), and each condition needs its own brackets.

In [ ]:
chain = pd.read_csv(SNAPSHOT)
chain["mid"] = (chain["bid"] + chain["ask"]) / 2

picked = chain[(chain["expiration"] == EXPIRY) & (chain["strike"] == STRIKE)]
picked[["option_type", "strike", "bid", "ask", "mid", "lastTradeDate"]]

You should see exactly two rows: one call and one put. Note that `lastTradeDate` is when the contract last **traded**, which is not when the bid/ask was quoted.

## Step 2 — The premium and the break-even points

The straddle premium is the call price plus the put price. A buyer breaks even if SPY finishes exactly one premium away from the strike, in either direction.

In [ ]:
call_mid = picked.loc[picked["option_type"] == "call", "mid"].item()
put_mid = picked.loc[picked["option_type"] == "put", "mid"].item()
premium = call_mid + put_mid

breakeven_low = STRIKE - premium
breakeven_high = STRIKE + premium

print(f"call mid {call_mid:.3f} + put mid {put_mid:.3f} = straddle premium {premium:.3f} per share")
print(f"per contract (x100 shares): ${premium * 100:,.0f}")
print(f"break-even at expiry: {breakeven_low:.2f} and {breakeven_high:.2f}")

## Step 3 — Plot value at expiry and profit, for the buyer and the seller

- **Value at expiry** (blue) is what the position is worth on Oct 23, before counting the premium. The buyer owns a V shape. The seller owes the same V, so it is flipped upside down.
- **Profit after premium** (orange) is the same line shifted by the premium: down for the buyer, who paid it, and up for the seller, who collected it.

In [ ]:
spots = np.linspace(720, 822, 400)
payoff = straddle_payoff(spots, STRIKE)

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7", "#fcfcfb"

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True, facecolor=SURFACE)

for ax, quantity, name in [(axes[0], +1, "Buyer (long 1 straddle)"),
                           (axes[1], -1, "Seller (short 1 straddle)")]:
    value = quantity * payoff
    profit = position_profit(payoff, premium, quantity)

    ax.set_facecolor(SURFACE)
    ax.axhline(0, color=AXIS, lw=1)
    ax.axvline(SPY_AT_SNAPSHOT, color=MUTED, lw=1, ls=":")
    ax.text(SPY_AT_SNAPSHOT + 1, -52, f"SPY on Sep 25\n{SPY_AT_SNAPSHOT:.2f}",
            color=MUTED, fontsize=8, va="bottom")

    ax.plot(spots, value, color=BLUE, lw=2, label="Value at expiry")
    ax.plot(spots, profit, color=ORANGE, lw=2, label="Profit after premium")

    for be in (breakeven_low, breakeven_high):
        ax.plot(be, 0, "o", ms=8, color=ORANGE, mec=SURFACE, mew=2, zorder=3)
        ax.annotate(f"{be:.2f}", (be, 0), xytext=(0, 12 * quantity + (-4 if quantity < 0 else 0)),
                    textcoords="offset points", ha="center", color=INK, fontsize=8)

    best = "max profit" if quantity < 0 else "max loss"
    ax.annotate(f"{best} {quantity * -premium:+.2f}\n(SPY exactly at {STRIKE:.0f})",
                (STRIKE, -quantity * premium), xytext=(0, -30 * quantity),
                textcoords="offset points", ha="center", va="center", color=INK, fontsize=8)

    ax.set_title(name, loc="left", color=INK, fontsize=11)
    ax.set_xlabel("SPY price at expiry, Oct 23 ($)", color=MUTED)
    ax.grid(color=GRID, lw=0.6)
    ax.tick_params(colors=MUTED, labelsize=8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)

axes[0].set_ylabel("$ per share  (x100 per contract)", color=MUTED)
axes[0].set_ylim(-55, 55)
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", ncol=2, frameon=False, fontsize=9, labelcolor=INK)
fig.suptitle(f"SPY {EXPIRY} {STRIKE:.0f} straddle at expiry, bought/sold at Sep 25 mid ({premium:.2f})",
             x=0.01, ha="left", color=INK, fontsize=12)
fig.text(0.01, -0.02, "Premium: observed Yahoo Finance bid/ask midpoints, 2026-09-25 16:14 ET. "
         "SPY expiry prices are hypothetical. Costs and early exercise ignored.",
         color=MUTED, fontsize=7.5)
fig.tight_layout()
fig.savefig("../reports/figures/01_straddle_payoff.png", dpi=150, bbox_inches="tight")
plt.show()

## Your turn

1. **Predict first, then check.** You thought SPY might rise about $10 by Oct 23, to 781.30. Read the seller's profit at 781.30 off the right-hand chart. Then compute it exactly in the cell below with `position_profit(straddle_payoff(781.30, STRIKE), premium, -1)`. Multiply by 100 to get dollars per contract.
2. **Change a parameter.** Set `STRIKE = 760.0` in the first code cell and re-run everything (**Run All**). The straddle is no longer at the money. What happens to the premium, and why is the V no longer centred on SPY's price?

In [ ]:
# 1. Your code here